# Google Open Knowledge Format (OKF) — Zero to Mastery

**A from-scratch, end-to-end implementation of Google's Open Knowledge Format (OKF v0.1),
solving a real problem: turning an undocumented, multi-table production dataset into an
agent-consumable knowledge base, plus a local, grounded question-answering agent over it.**

Everything in this notebook runs for real, locally:

- **Dataset**: [Olist Brazilian E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
  — real, anonymized, production e-commerce data (~100k orders, 9 relational tables) downloaded
  live from Kaggle by the code below.
- **Models**: local Ollama models, both ≤ 4B parameters — `qwen3.5:4b` (generation/reasoning-off)
  and `nomic-embed-text` (embeddings). No cloud APIs, no API keys beyond Kaggle.
- **OKF implementation**: written from scratch against the [official v0.1 specification]
  (https://github.com/GoogleCloudPlatform/knowledge-catalog/blob/main/okf/SPEC.md) — frontmatter
  parsing/rendering, bundle structure, index/log generation, cross-linking, and a real
  conformance validator.

## Roadmap

1. **Theory** — what OKF is, why it exists, the full v0.1 spec explained with a worked example.
2. **The real problem** — an undocumented production dataset, and why agents can't use it as-is.
3. **Data acquisition & profiling** — download the dataset, profile every table for real.
4. **OKF core library, from scratch** — documents, frontmatter, bundles, index/log, conformance.
5. **Local LLM enrichment agent** — a grounded producer that drafts OKF concept docs from the
   profiled data using a local 4B model (facts are computed in Python; the LLM only writes prose).
6. **Conformance validation** — prove the generated bundle actually satisfies the spec.
7. **Visualization** — render the knowledge graph as a self-contained interactive HTML file.
8. **Local RAG discovery agent** — a grounded consumer: embed → retrieve → LLM-rerank → cited answer.
9. **Evaluation** — retrieval accuracy and answer groundedness, measured honestly, no cherry-picking.
10. **Mastery recap** — what we built vs. Google's reference implementation, limitations, references.

By the end you will have *produced* and *consumed* a real OKF bundle end-to-end, and understand
every rule in the spec well enough to write a producer or consumer for your own data.

## 0. Environment check

We verify the local model runtime before doing anything else — every LLM call downstream depends
on these two models being present and responsive.

In [1]:
import json
import os
import re
import subprocess
import time
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
from itertools import groupby
from pathlib import Path
from typing import Any, Optional

import faiss
import numpy as np
import ollama
import pandas as pd
import yaml
from loguru import logger
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

GEN_MODEL = "qwen3.5:4b"
EMBED_MODEL = "nomic-embed-text"
BUNDLE_ROOT = Path("bundle")
RUN_TS = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

logger.remove()
logger.add(lambda msg: print(msg, end=""), level="INFO", format="<level>{message}</level>\n")

1

In [2]:
local_models = {m["model"] for m in ollama.list()["models"]}


def model_available(name: str) -> bool:
    # `ollama list` returns tagged names (e.g. "nomic-embed-text:latest"); match with or without a tag.
    return name in local_models or any(m.split(":")[0] == name.split(":")[0] for m in local_models)


for required in (GEN_MODEL, EMBED_MODEL):
    assert model_available(required), (
        f"Missing local model {required!r}. Pull it with `ollama pull {required}` first."
    )
print("Available and ready:")
print(f"  generation model : {GEN_MODEL}")
print(f"  embedding model  : {EMBED_MODEL}")
print(f"  run timestamp    : {RUN_TS}")

Available and ready:
  generation model : qwen3.5:4b
  embedding model  : nomic-embed-text
  run timestamp    : 2026-07-05T08:05:19+00:00


In [3]:
def llm(prompt: str, *, system: Optional[str] = None, num_predict: int = 350,
        temperature: float = 0.2, retries: int = 2) -> str:
    """Call the local generation model with thinking disabled (qwen3.5 is a hybrid-reasoning
    model; without `think=False` it can burn its entire token budget on hidden <think> traces
    and return empty content — verified during development of this notebook).
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    last_err = None
    for attempt in range(retries + 1):
        try:
            resp = ollama.chat(
                model=GEN_MODEL,
                messages=messages,
                think=False,
                options={"num_predict": num_predict, "temperature": temperature},
            )
            content = resp["message"]["content"].strip()
            if content:
                return content
        except Exception as e:  # pragma: no cover - network/runtime hiccups
            last_err = e
        time.sleep(0.5)
    raise RuntimeError(f"LLM call failed after {retries + 1} attempts: {last_err}")


def embed(text: str) -> np.ndarray:
    resp = ollama.embed(model=EMBED_MODEL, input=text)
    return np.array(resp["embeddings"][0], dtype="float32")


# smoke test
print(llm("Reply with exactly the word: ready"))

ready


## 1. Theory — What is the Open Knowledge Format?

### 1.1 The problem it solves

Organizations accumulate *knowledge about their data* — what a table means, how it joins to
others, how a metric is defined, what a playbook says to do during an incident — in wildly
incompatible places: proprietary catalog UIs, wikis, code comments, Slack threads, and senior
engineers' heads. When an AI agent is asked *"how do I compute weekly active users from our
event stream?"*, it has no single place to look, and no consistent format even if it did.

Google's answer, published in the `GoogleCloudPlatform/knowledge-catalog` repository, is **not**
another service or database. It is a **format**: the Open Knowledge Format (OKF), v0.1 draft.

### 1.2 The core idea

> OKF is an open, human- and agent-friendly format for representing *knowledge* — the metadata,
> context, and curated insight that surrounds data and systems.

Concretely: **a directory of Markdown files with YAML frontmatter.** That's the entire mechanism.
No schema registry, no required SDK, no runtime. If you can `cat` a file, you can read OKF. If
you can `git clone` a repo, you can ship it.

This is a deliberate design philosophy — a **format, not a platform**:

| Property | Why it matters |
|---|---|
| Readable by humans without tooling | An engineer can open the file in any editor |
| Parseable by agents without bespoke SDKs | Any LLM can ingest it verbatim into context |
| Diffable in version control | Knowledge curation becomes a normal engineering activity (PRs, blame, review) |
| Portable across tools, orgs, and time | A bundle is just a directory — ship it as a tarball, host it in any repo |

### 1.3 Terminology (SPEC.md §2)

| Term | Meaning |
|---|---|
| **Knowledge Bundle** | A self-contained, hierarchical collection of knowledge documents — the unit of distribution |
| **Concept** | A single unit of knowledge, represented as one Markdown file. Can describe a tangible asset (a table, an API) or an abstract idea (a metric, a playbook) |
| **Concept ID** | The concept's file path within the bundle, with `.md` removed (e.g. `tables/users.md` → `tables/users`) |
| **Frontmatter** | The YAML metadata block delimited by `---` at the top of a file |
| **Body** | Everything after the frontmatter — free-form Markdown |
| **Link** | A standard Markdown link from one concept to another, expressing a relationship |
| **Citation** | A link from a concept to an external source backing a claim in the body |

### 1.4 Bundle structure (SPEC.md §3)

```
path/to/bundle/
├── index.md                # optional — directory listing, progressive disclosure
├── log.md                  # optional — chronological change history
├── <concept>.md            # a concept at the bundle root
└── <subdirectory>/         # subdirectories group concepts however the producer wants
    ├── index.md
    ├── <concept>.md
    └── <subdirectory>/…
```

`index.md` and `log.md` are **reserved filenames** — every other `.md` file is a concept document.

### 1.5 Concept documents (SPEC.md §4)

Only **one** frontmatter field is required: `type`. Everything else is recommended-but-optional,
and producers may add arbitrary extra keys that consumers must tolerate.

```yaml
---
type: <Type name>                  # REQUIRED — e.g. "BigQuery Table", "Metric", "Playbook"
title: <display name>              # recommended
description: <one-line summary>    # recommended
resource: <canonical URI>          # recommended, for concepts bound to a physical asset
tags: [<tag>, …]                   # optional
timestamp: <ISO 8601 datetime>     # optional, last-modified time
---
```

The body is free Markdown. Three section headings carry **conventional** (not required) meaning:
`# Schema`, `# Examples`, `# Citations`.

### 1.6 Cross-linking (SPEC.md §5)

Concepts link to each other with ordinary Markdown links. **Bundle-absolute** links
(`[text](/tables/customers.md)`, resolved from the bundle root) are the recommended form because
they stay valid when a file moves within its own subdirectory. The *meaning* of a link (parent/
child, joins-with, depends-on…) lives in the surrounding prose — a link is just an untyped edge.
Consumers **must tolerate broken links**: a link to a concept that doesn't exist yet simply
represents not-yet-written knowledge, not a malformed document.

### 1.7 Index & log files (SPEC.md §6–7)

`index.md` has **no frontmatter** and lists a directory's concepts grouped under headings, for
*progressive disclosure* — an agent can see what exists one level at a time instead of loading an
entire bundle into context. `log.md` records dated, newest-first change entries
(`**Update**`, `**Creation**`, `**Deprecation**` are conventional lead words, not requirements).

### 1.8 Conformance (SPEC.md §9) — the permissive-consumption model

A bundle is conformant if and only if:

1. Every non-reserved `.md` file has a parseable YAML frontmatter block.
2. Every frontmatter block has a non-empty `type` field.
3. Reserved filenames (`index.md`, `log.md`) follow their documented structure when present.

Everything else — missing optional fields, unknown `type` values, unknown extra keys, **broken
links**, missing `index.md` — is soft guidance. Consumers **must not** reject a bundle over these.
This is deliberate: OKF must stay useful as bundles grow, get refactored, and are partially
generated by agents mid-flight.

### 1.9 A minimal worked example

Before building the full library, let's hand-write one conformant concept and parse it back —
proving we understand the two mechanical rules (§9.1 and §9.2) with real code, not just prose.

In [4]:
mini_example = """---
type: Metric
title: Average Order Value
description: Mean total payment value per order.
tags: [metrics, revenue]
timestamp: 2026-01-01T00:00:00+00:00
---

The average order value is the mean of total payments across all orders.

# Citations

[1] Computed from the dataset in this notebook.
"""

fm_block, body = re.match(r"^---\n(.*?)\n---\n(.*)$", mini_example, re.DOTALL).groups()
frontmatter = yaml.safe_load(fm_block)
print("Parsed frontmatter:", frontmatter)
assert frontmatter.get("type"), "Rule §9.2 violated: 'type' must be non-empty"
print("\nConformant: type is present and non-empty ->", bool(frontmatter.get("type")))

Parsed frontmatter: {'type': 'Metric', 'title': 'Average Order Value', 'description': 'Mean total payment value per order.', 'tags': ['metrics', 'revenue'], 'timestamp': datetime.datetime(2026, 1, 1, 0, 0, tzinfo=datetime.timezone.utc)}

Conformant: type is present and non-empty -> True


## 2. The real problem

The [Olist Brazilian E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
is real, anonymized order data from a real Brazilian marketplace, split across **nine CSV files**
with foreign-key relationships that are documented *nowhere in the files themselves* — exactly
the situation OKF targets: tribal knowledge about a real dataset, currently readable only by
whoever last worked with it.

We will act as both the **producer** (an enrichment agent that walks the dataset and drafts an
OKF bundle, mirroring what Google's reference agent does for BigQuery) and the **consumer** (a
retrieval agent that answers real questions using nothing but the bundle we produced).

### 2.1 Download the dataset (from code, live from Kaggle)

In [5]:
import kagglehub

DATASET_DIR = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
csv_files = sorted(DATASET_DIR.glob("*.csv"))
print(f"Downloaded to: {DATASET_DIR}")
for f in csv_files:
    print(f"  {f.name:45s} {f.stat().st_size / 1e6:6.2f} MB")

Downloaded to: /home/ahmad/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2
  olist_customers_dataset.csv                     9.03 MB
  olist_geolocation_dataset.csv                  61.27 MB
  olist_order_items_dataset.csv                  15.44 MB
  olist_order_payments_dataset.csv                5.78 MB
  olist_order_reviews_dataset.csv                14.45 MB
  olist_orders_dataset.csv                       17.65 MB
  olist_products_dataset.csv                      2.38 MB
  olist_sellers_dataset.csv                       0.17 MB
  product_category_name_translation.csv           0.00 MB


### 2.2 Load every table

Table names are derived from the CSV filenames (stripping the `olist_`/`_dataset` boilerplate)
so the OKF concept IDs read naturally, e.g. `tables/orders` rather than `tables/olist_orders_dataset`.

In [6]:
TABLE_RENAME = {"product_category_name_translation": "category_translation"}


def table_name_from_file(path: Path) -> str:
    name = path.stem
    name = re.sub(r"^olist_", "", name)
    name = re.sub(r"_dataset$", "", name)
    return TABLE_RENAME.get(name, name)


tables: dict[str, pd.DataFrame] = {}
for f in csv_files:
    name = table_name_from_file(f)
    tables[name] = pd.read_csv(f)

print(f"Loaded {len(tables)} tables:\n")
for name, df in tables.items():
    print(f"  {name:35s} {df.shape[0]:>8,d} rows  x {df.shape[1]:>2d} cols")

Loaded 9 tables:

  customers                             99,441 rows  x  5 cols
  geolocation                         1,000,163 rows  x  5 cols
  order_items                          112,650 rows  x  7 cols
  order_payments                       103,886 rows  x  5 cols
  order_reviews                         99,224 rows  x  7 cols
  orders                                99,441 rows  x  8 cols
  products                              32,951 rows  x  9 cols
  sellers                                3,095 rows  x  4 cols
  category_translation                      71 rows  x  2 cols


### 2.3 Real EDA — schema, nulls, and samples for every table

This is the *only* place facts are allowed to originate in this notebook: everything the LLM
writes about the data later must trace back to numbers computed here.

In [7]:
def profile_table(name: str, df: pd.DataFrame) -> dict:
    columns = []
    for col in df.columns:
        s = df[col]
        samples = [str(v) for v in s.dropna().unique()[:3]]
        columns.append({
            "name": col,
            "dtype": str(s.dtype),
            "null_pct": round(float(s.isna().mean()) * 100, 2),
            "n_unique": int(s.nunique()),
            "samples": samples,
        })
    return {"name": name, "n_rows": int(len(df)), "n_cols": int(len(df.columns)), "columns": columns}


profiles = {name: profile_table(name, df) for name, df in tables.items()}

overview_rows = []
for name, p in profiles.items():
    overview_rows.append({
        "table": name,
        "rows": p["n_rows"],
        "cols": p["n_cols"],
        "avg_null_pct": round(np.mean([c["null_pct"] for c in p["columns"]]), 2),
    })
display(pd.DataFrame(overview_rows).sort_values("rows", ascending=False).reset_index(drop=True))

,table,rows,cols,avg_null_pct
0,geolocation,1000163,5,0.00
1,order_items,112650,7,0.00
2,order_payments,103886,5,0.00
3,customers,99441,5,0.00
4,orders,99441,8,0.62
5,order_reviews,99224,7,21.01
6,products,32951,9,0.83
7,sellers,3095,4,0.00
8,category_translation,71,2,0.00


In [8]:
# A closer look at one table, to see exactly what the enrichment agent will be grounded on.
pd.DataFrame(profiles["orders"]["columns"])

,name,dtype,null_pct,n_unique,samples
0,order_id,str,0.00,99441,"[e481f51cbdc54678b7cc49136f2d6af7, 53cdb2fc8bc..."
1,customer_id,str,0.00,99441,"[9ef432eb6251297304e76186b10a928d, b0830fb4747..."
2,order_status,str,0.00,8,"[delivered, invoiced, shipped]"
3,order_purchase_timestamp,str,0.00,98875,"[2017-10-02 10:56:33, 2018-07-24 20:41:37, 201..."
4,order_approved_at,str,0.16,90733,"[2017-10-02 11:07:15, 2018-07-26 03:24:27, 201..."
5,order_delivered_carrier_date,str,1.79,81018,"[2017-10-04 19:55:00, 2018-07-26 14:31:00, 201..."
6,order_delivered_customer_date,str,2.98,95664,"[2017-10-10 21:25:13, 2018-08-07 15:27:45, 201..."
7,order_estimated_delivery_date,str,0.00,459,"[2017-10-18 00:00:00, 2018-08-13 00:00:00, 201..."


## 3. The OKF core library — built from scratch

We now implement OKF v0.1 directly against the spec sections above: a document model, frontmatter
rendering/parsing, link extraction, bundle I/O, index/log generation, and a conformance validator.
This is an independent implementation (not a copy of Google's reference agent) — the point is to
*understand* the format well enough to produce and consume it with nothing but the spec.

### 3.1 The concept document model (SPEC.md §4.1)

In [9]:
RESERVED_FILENAMES = {"index.md", "log.md"}


@dataclass
class OkfDocument:
    """One OKF concept document: a required `type`, recommended metadata, and a free-form body."""

    type: str
    title: Optional[str] = None
    description: Optional[str] = None
    resource: Optional[str] = None
    tags: list[str] = field(default_factory=list)
    timestamp: Optional[str] = None
    extra: dict[str, Any] = field(default_factory=dict)
    body: str = ""

    def frontmatter_dict(self) -> dict:
        assert self.type, "SPEC.md §9.2: 'type' is a required, non-empty frontmatter field"
        fm: dict[str, Any] = {"type": self.type}
        if self.title:
            fm["title"] = self.title
        if self.description:
            fm["description"] = self.description
        if self.resource:
            fm["resource"] = self.resource
        if self.tags:
            fm["tags"] = self.tags
        if self.timestamp:
            fm["timestamp"] = self.timestamp
        fm.update(self.extra)
        return fm

    def render(self) -> str:
        fm_yaml = yaml.safe_dump(
            self.frontmatter_dict(), sort_keys=False, allow_unicode=True, default_flow_style=False
        ).strip()
        return f"---\n{fm_yaml}\n---\n\n{self.body.strip()}\n"


FRONTMATTER_RE = re.compile(r"^---\n(.*?)\n---\n(.*)$", re.DOTALL)


def parse_concept(path: Path) -> tuple[dict, str]:
    """Parse a concept file into (frontmatter dict, body). Raises if no parseable block exists —
    this is the one hard structural rule a *producer* must satisfy (SPEC.md §9.1)."""
    text = path.read_text(encoding="utf-8")
    m = FRONTMATTER_RE.match(text)
    if not m:
        raise ValueError(f"{path}: no parseable YAML frontmatter block (SPEC.md §9.1 violation)")
    fm = yaml.safe_load(m.group(1)) or {}
    return fm, m.group(2)


def write_concept(bundle_root: Path, concept_id: str, doc: OkfDocument) -> Path:
    if Path(f"{concept_id}.md").name in RESERVED_FILENAMES:
        raise ValueError(f"{concept_id!r} collides with a reserved filename")
    path = bundle_root / f"{concept_id}.md"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(doc.render(), encoding="utf-8")
    return path


def iter_concept_files(bundle_root: Path):
    for p in sorted(bundle_root.rglob("*.md")):
        if p.name not in RESERVED_FILENAMES:
            yield p

**Round-trip self-test** — write a document, parse it back, and check nothing was lost. This is
the smallest possible proof that the render/parse pair is correct before we trust it with 20+
LLM-generated documents later.

In [10]:
_test_dir = Path("_okf_selftest")
_test_doc = OkfDocument(
    type="Metric", title="Test Metric", description="A round-trip test.",
    tags=["test"], timestamp=RUN_TS, body="# Citations\n\n[1] N/A",
)
_test_path = write_concept(_test_dir, "metrics/test", _test_doc)
_fm, _body = parse_concept(_test_path)
assert _fm["type"] == "Metric" and _fm["title"] == "Test Metric" and "Citations" in _body
print("Round-trip OK:", _fm)
import shutil
shutil.rmtree(_test_dir)

Round-trip OK: {'type': 'Metric', 'title': 'Test Metric', 'description': 'A round-trip test.', 'tags': ['test'], 'timestamp': '2026-07-05T08:05:19+00:00'}


### 3.2 Cross-linking (SPEC.md §5)

We only need to *extract* links (to build the graph and check conformance) — the spec assigns no
semantics to a link beyond "a relationship exists," so no further parsing is required.

In [11]:
LINK_RE = re.compile(r"\[([^\]]+)\]\(([^)]+)\)")


def extract_links(body: str) -> list[dict]:
    links = []
    for text, target in LINK_RE.findall(body):
        if target.startswith(("http://", "https://")):
            kind = "external"
        elif target.startswith("/"):
            kind = "bundle-absolute"
        else:
            kind = "relative"
        links.append({"text": text, "target": target, "kind": kind})
    return links


def concept_id_for(path: Path, bundle_root: Path) -> str:
    return str(path.relative_to(bundle_root).with_suffix("")).replace(os.sep, "/")


def resolve_link_target(link: dict, from_concept_id: str) -> Optional[str]:
    """Best-effort resolution of a link to a concept ID, for graph-building and link-checking."""
    if link["kind"] == "external":
        return None
    if link["kind"] == "bundle-absolute":
        target = link["target"].lstrip("/")
    else:  # relative
        target = str((Path(from_concept_id).parent / link["target"]).as_posix())
    return re.sub(r"\.md$", "", target)

### 3.3 Index files (SPEC.md §6) — progressive disclosure

`index.md` has **no frontmatter** (except, uniquely, an optional `okf_version` key at the bundle
root — SPEC.md §11). We synthesize it by grouping each directory's concepts by their `type`.

In [12]:
def generate_index(dir_path: Path, bundle_root: Path, *, okf_version: Optional[str] = None,
                    title: Optional[str] = None) -> str:
    groups: dict[str, list[tuple[str, str, str]]] = defaultdict(list)
    for child in sorted(dir_path.iterdir()):
        if child.name in RESERVED_FILENAMES:
            continue
        if child.is_dir():
            groups["Subdirectories"].append((child.name + "/", child.name + "/", f"{child.name} concepts"))
        elif child.suffix == ".md":
            fm, _ = parse_concept(child)
            child_title = fm.get("title", child.stem)
            desc = fm.get("description", "")
            groups[fm.get("type", "Concept")].append((child_title, child.name, desc))

    lines = []
    if okf_version:
        lines += ["---", f"okf_version: \"{okf_version}\"", "---", ""]
    lines.append(f"# {title or dir_path.name or bundle_root.name}\n")
    for group, entries in groups.items():
        lines.append(f"# {group}\n")
        for title, href, desc in entries:
            lines.append(f"* [{title}]({href})" + (f" - {desc}" if desc else ""))
        lines.append("")
    return "\n".join(lines).strip() + "\n"


def write_index(dir_path: Path, bundle_root: Path, **kwargs) -> Path:
    text = generate_index(dir_path, bundle_root, **kwargs)
    path = dir_path / "index.md"
    path.write_text(text, encoding="utf-8")
    return path

### 3.4 Log files (SPEC.md §7)

In [13]:
def generate_log(entries: list[dict]) -> str:
    lines = ["# Directory Update Log\n"]
    entries = sorted(entries, key=lambda e: e["date"], reverse=True)
    for date, group in groupby(entries, key=lambda e: e["date"]):
        lines.append(f"## {date}")
        for e in group:
            lines.append(f"* **{e['kind']}**: {e['text']}")
        lines.append("")
    return "\n".join(lines).strip() + "\n"

### 3.5 Conformance validator (SPEC.md §9)

This is the crux of the "permissive consumption" model: exactly two hard rules, everything else
reported as a soft warning that must **not** cause rejection.

In [14]:
def validate_conformance(bundle_root: Path) -> dict:
    errors, warnings_, concept_ids = [], [], []
    parsed: dict[str, tuple[dict, str]] = {}

    for p in iter_concept_files(bundle_root):
        cid = concept_id_for(p, bundle_root)
        concept_ids.append(cid)
        try:
            fm, body = parse_concept(p)
        except ValueError as e:
            errors.append(str(e))
            continue
        if not fm.get("type"):
            errors.append(f"{cid}: missing/empty required 'type' field (SPEC.md §9.2)")
        parsed[cid] = (fm, body)

    id_set = set(concept_ids)
    for cid, (fm, body) in parsed.items():
        for link in extract_links(body):
            target = resolve_link_target(link, cid)
            if target is not None and target not in id_set:
                warnings_.append(f"{cid}: broken link -> {link['target']} (tolerated per SPEC.md §9)")

    for reserved in RESERVED_FILENAMES:
        for p in bundle_root.rglob(reserved):
            if reserved == "index.md" and p.read_text(encoding="utf-8").strip() == "":
                warnings_.append(f"{p.relative_to(bundle_root)}: empty index.md")

    return {
        "conformant": len(errors) == 0,
        "n_concepts": len(concept_ids),
        "n_errors": len(errors),
        "n_warnings": len(warnings_),
        "errors": errors,
        "warnings": warnings_,
    }

## 4. Foreign-key discovery — deterministic, not the LLM's job

Before any generation happens, we discover real join relationships with plain pandas: for every
pair of same-named columns across tables (normalizing `*_zip_code_prefix` variants, since
`customer_zip_code_prefix` and `seller_zip_code_prefix` both join to `geolocation_zip_code_prefix`
under a different name), we measure what fraction of one column's values are contained in the
other. High overlap (≥ 90%) is treated as a foreign-key candidate.

**Why this matters for OKF**: the "Joins" section of every table concept below is 100%
programmatic. The LLM never invents a relationship — it only narrates ones we've already proven
exist in the data. Facts come from code; the model supplies prose.

In [15]:
def base_key(col: str) -> str:
    return "zip_code_prefix" if col.endswith("zip_code_prefix") else col


def detect_fk_candidates(tables: dict[str, pd.DataFrame], min_overlap: float = 0.9) -> pd.DataFrame:
    groups: dict[str, list[tuple[str, str]]] = defaultdict(list)
    for tname, df in tables.items():
        for col in df.columns:
            groups[base_key(col)].append((tname, col))

    rows = []
    for members in groups.values():
        if len(members) < 2:
            continue
        for i, (t1, c1) in enumerate(members):
            for j, (t2, c2) in enumerate(members):
                if i == j:
                    continue
                s1, s2 = tables[t1][c1].dropna(), tables[t2][c2].dropna()
                if s1.empty or s2.empty:
                    continue
                overlap = float(s1.isin(set(s2.unique())).mean())
                if overlap >= min_overlap:
                    rows.append({
                        "from_table": t1, "from_column": c1, "to_table": t2, "to_column": c2,
                        "overlap": round(overlap, 4),
                        "from_cardinality": int(s1.nunique()), "to_cardinality": int(s2.nunique()),
                    })
    return pd.DataFrame(rows)


fk_candidates = detect_fk_candidates(tables)
display(fk_candidates.sort_values(["from_table", "overlap"], ascending=[True, False]).reset_index(drop=True))

,from_table,from_column,to_table,to_column,overlap,from_cardinality,to_cardinality
0,category_translation,product_category_name,products,product_category_name,1.0000,71,73
1,customers,customer_id,orders,customer_id,1.0000,99441,99441
2,customers,customer_zip_code_prefix,geolocation,geolocation_zip_code_prefix,0.9972,14994,19015
3,geolocation,geolocation_zip_code_prefix,customers,customer_zip_code_prefix,0.9681,19015,14994
4,order_items,order_id,order_payments,order_id,1.0000,98666,99440
5,order_items,order_id,orders,order_id,1.0000,98666,99441
6,order_items,product_id,products,product_id,1.0000,32951,32951
7,order_items,seller_id,sellers,seller_id,1.0000,3095,3095
8,order_items,order_id,order_reviews,order_id,0.9916,98666,98673
9,order_payments,order_id,orders,order_id,1.0000,99440,99441


This heuristic is deliberately simple and, as a result, honest about its limits: it only catches
relationships expressed through matching (or `zip_code_prefix`-normalized) column names. A join
based on business logic with no shared column name — e.g. matching orders to marketing campaigns
by date range — would be invisible to it. That gap is exactly what a human curator (or a
web-crawling enrichment pass, like Google's reference agent's second pass) is for; OKF's citation
and Notes sections exist to capture precisely this kind of knowledge that code can't discover on
its own.

## 5. Real metrics — computed once, cited everywhere

Six metrics, each computed directly from the data with pandas. These numbers are the ground truth
that the LLM's prose will be checked against later.

In [16]:
orders_df = tables["orders"].copy()
for col in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders_df[col] = pd.to_datetime(orders_df[col], errors="coerce")

payments_per_order = tables["order_payments"].groupby("order_id")["payment_value"].sum()
avg_order_value = float(payments_per_order.mean())

delivered = orders_df[orders_df["order_status"] == "delivered"].dropna(
    subset=["order_delivered_customer_date", "order_estimated_delivery_date"]
)
late_mask = delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
late_delivery_rate = float(late_mask.mean())

review_scores = tables["order_reviews"]["review_score"]
avg_review_score = float(review_scores.mean())
review_score_distribution = (review_scores.value_counts(normalize=True).sort_index() * 100).round(2)

payment_type_distribution = (
    tables["order_payments"]["payment_type"].value_counts(normalize=True) * 100
).round(2)

cat_en = tables["products"].merge(tables["category_translation"], on="product_category_name", how="left")
top_categories = (
    tables["order_items"]
    .merge(cat_en[["product_id", "product_category_name_english"]], on="product_id", how="left")
    .groupby("product_category_name_english")["order_id"].nunique()
    .sort_values(ascending=False).head(5)
)

metrics_summary = {
    "avg_order_value": {
        "value": round(avg_order_value, 2), "unit": "BRL",
        "n": int(payments_per_order.shape[0]),
        "source_tables": ["order_payments"],
        "definition": "mean(sum(payment_value) grouped by order_id)",
    },
    "late_delivery_rate": {
        "value": round(late_delivery_rate * 100, 2), "unit": "%",
        "n": int(len(delivered)),
        "source_tables": ["orders"],
        "definition": "mean(order_delivered_customer_date > order_estimated_delivery_date) over delivered orders",
    },
    "avg_review_score": {
        "value": round(avg_review_score, 2), "unit": "stars (1-5)",
        "n": int(review_scores.notna().sum()),
        "source_tables": ["order_reviews"],
        "definition": "mean(review_score)",
    },
    "review_score_distribution": {
        "value": review_score_distribution.to_dict(), "unit": "% of reviews",
        "n": int(review_scores.notna().sum()),
        "source_tables": ["order_reviews"],
        "definition": "value_counts(review_score, normalize=True)",
    },
    "payment_type_distribution": {
        "value": payment_type_distribution.to_dict(), "unit": "% of payments",
        "n": int(tables["order_payments"].shape[0]),
        "source_tables": ["order_payments"],
        "definition": "value_counts(payment_type, normalize=True)",
    },
    "top_product_categories": {
        "value": top_categories.to_dict(), "unit": "distinct orders",
        "n": int(top_categories.sum()),
        "source_tables": ["order_items", "products", "category_translation"],
        "definition": "nunique(order_id) grouped by product_category_name_english, top 5",
    },
}

for k, v in metrics_summary.items():
    print(f"{k:28s} = {v['value']}  {v['unit']}  (n={v['n']:,})")

avg_order_value              = 160.99  BRL  (n=99,440)
late_delivery_rate           = 8.11  %  (n=96,470)
avg_review_score             = 4.09  stars (1-5)  (n=99,224)
review_score_distribution    = {1: 11.51, 2: 3.18, 3: 8.24, 4: 19.29, 5: 57.78}  % of reviews  (n=99,224)
payment_type_distribution    = {'credit_card': 73.92, 'boleto': 19.04, 'voucher': 5.56, 'debit_card': 1.47, 'not_defined': 0.0}  % of payments  (n=103,886)
top_product_categories       = {'bed_bath_table': 9417, 'health_beauty': 8836, 'sports_leisure': 7720, 'computers_accessories': 6689, 'furniture_decor': 6449}  distinct orders  (n=39,111)


## 6. The local LLM enrichment agent

This mirrors what Google's reference agent does for BigQuery (SPEC.md's motivating example),
except the source is pandas instead of BigQuery metadata and the model is a local 4B model
instead of Gemini. The division of labor is the important part, and it holds regardless of
source or model:

- **Code computes every fact**: schema, null rates, foreign keys, metric values.
- **The LLM only writes prose**: descriptions and narrative notes, grounded in the facts it's
  handed — never asked to recall or invent a number on its own.
- **Every generated number is checked against ground truth** before being accepted.

### 6.1 Formatting helpers (programmatic — no LLM involved)

In [17]:
KAGGLE_URL = "https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce"


def format_schema_table(profile: dict) -> str:
    lines = ["| Column | Type | Null % | Unique | Sample values |", "|---|---|---|---|---|"]
    for c in profile["columns"]:
        samples = ", ".join(f"`{s[:30]}`" for s in c["samples"]) or "*(none)*"
        lines.append(f"| `{c['name']}` | {c['dtype']} | {c['null_pct']}% | {c['n_unique']:,} | {samples} |")
    return "\n".join(lines)


def format_joins_section(table: str, fk_df: pd.DataFrame) -> str:
    outgoing = fk_df[fk_df["from_table"] == table]
    if outgoing.empty:
        return "No outgoing foreign-key relationships were detected from this table."
    lines = []
    for _, r in outgoing.iterrows():
        lines.append(
            f"- `{r['from_column']}` -> [{r['to_table']}](/tables/{r['to_table']}.md) "
            f"(`{r['overlap'] * 100:.1f}%` value overlap, "
            f"{r['from_cardinality']:,} -> {r['to_cardinality']:,} distinct values)"
        )
    return "\n".join(lines)


def first_sentence(text: str, max_len: int = 220) -> str:
    """Frontmatter `description` should be a clean one-liner. Blind character slicing cuts mid-word
    (a real bug caught during development: '...capturing end-to-end data from produc' — truncated
    mid-"product"); breaking on a sentence boundary first avoids that."""
    text = text.strip()
    m = re.search(r".+?[.!?](?=\s|$)", text)
    sentence = m.group(0) if m else text
    if len(sentence) > max_len:
        sentence = sentence[:max_len].rsplit(" ", 1)[0].rstrip(".,;:") + "…"
    return sentence


def format_metric_value(info: dict) -> str:
    v = info["value"]
    if isinstance(v, dict):
        return ", ".join(f"{k}: {v2}" for k, v2 in v.items())
    return str(v)


def number_variants(x: float) -> set[str]:
    return {f"{x}", f"{x:.0f}", f"{x:.1f}", f"{x:.2f}", str(round(x))}


def metric_is_grounded(text: str, info: dict) -> bool:
    """A generated metric explanation is 'grounded' if at least one of the real computed numbers
    appears verbatim in the text. This is a cheap but real faithfulness check — exactly the kind
    of thing that matters once an LLM is writing documentation nobody will manually fact-check."""
    clean = text.replace(",", "")
    v = info["value"]
    candidates = v.values() if isinstance(v, dict) else [v]
    return any(variant in clean for x in candidates for variant in number_variants(float(x)))

### 6.2 Prompt templates

Every prompt states the facts up front and explicitly forbids invention. This is the same
grounding discipline used for RAG answer generation later in Part 8 — worth internalizing once.

In [18]:
def build_table_prompt(name: str, profile: dict, fk_df: pd.DataFrame) -> str:
    outgoing = fk_df[fk_df["from_table"] == name]
    incoming = fk_df[fk_df["to_table"] == name]
    rel_bits = [f"{name}.{r.from_column} -> {r.to_table}.{r.to_column} ({r.overlap * 100:.0f}% overlap)"
                for r in outgoing.itertuples()]
    rel_bits += [f"{r.from_table}.{r.from_column} -> {name}.{r.to_column} ({r.overlap * 100:.0f}% overlap)"
                 for r in incoming.itertuples()]
    col_summary = "; ".join(f"{c['name']} ({c['dtype']}, {c['null_pct']}% null)" for c in profile["columns"])
    return f"""You are documenting a database table for a knowledge base read by engineers and AI agents.
Use ONLY the facts given below. Do not invent business meaning beyond what the column names imply, and never invent numbers.

Table name: {name}
Row count: {profile['n_rows']:,}
Columns: {col_summary}
Detected relationships: {'; '.join(rel_bits) if rel_bits else 'none detected'}

Write exactly two short paragraphs separated by a single blank line:
1. A 1-2 sentence description of what one row in this table represents.
2. A 1-2 sentence note mentioning any relationships to other tables and any notable data-quality
   signal (e.g. a column with a high null percentage), using only the facts above.

Output ONLY the two paragraphs as plain prose. No headers, no markdown, no preamble."""


def build_metric_prompt(key: str, info: dict) -> str:
    value_str = format_metric_value(info)
    return f"""You are documenting a business metric for a knowledge base.

Metric: {key.replace('_', ' ')}
Computed value: {value_str} {info['unit']}
Computed over: {info['n']:,} records
Definition: {info['definition']}
Source table(s): {', '.join(info['source_tables'])}

Write a 2-3 sentence plain-English explanation of what this metric measures and what its computed
value means in practice. You MUST include the exact computed value ({value_str}) verbatim somewhere
in your text. Output ONLY the prose — no headers, no preamble."""


def build_dataset_prompt(profiles: dict, date_range: str) -> str:
    """Includes the real order date range explicitly. Without it, a first run of this exact prompt
    produced a confident, plausible-sounding, and completely wrong '...between 2019 and 2023' —
    the model filling a gap we hadn't grounded. Handing it the true range (2016-09-04 to
    2018-10-17) removes the incentive to guess."""
    table_list = "\n".join(f"- {n}: {p['n_rows']:,} rows, {p['n_cols']} columns" for n, p in profiles.items())
    return f"""You are documenting an entire dataset for a knowledge base.

Dataset: Olist Brazilian E-Commerce Public Dataset
Order date range: {date_range}
Tables:
{table_list}

Write a 2-3 sentence overview of what this dataset represents as a whole and what kind of business
questions it can answer. Use ONLY the facts given above — in particular, do not state or imply any
date range other than the one given. Output ONLY the prose, no headers, no preamble."""

### 6.3 The three enrichment functions

Each returns a fully-formed `OkfDocument` — Schema, Joins, and Computed Value sections are
assembled in Python; only the lead paragraph(s) come from the model.

In [19]:
def enrich_table(name: str) -> OkfDocument:
    profile = profiles[name]
    raw = llm(build_table_prompt(name, profile, fk_candidates), num_predict=250)
    parts = raw.split("\n\n", 1)
    description = parts[0].strip()
    notes = parts[1].strip() if len(parts) > 1 else "No additional notes generated."
    body = f"""{description}

# Schema

{format_schema_table(profile)}

# Joins

{format_joins_section(name, fk_candidates)}

# Notes

{notes}

# Citations

[1] [Olist Brazilian E-Commerce Public Dataset (Kaggle)]({KAGGLE_URL}) — table `{name}`."""
    return OkfDocument(
        type="Table", title=name.replace("_", " ").title(), description=first_sentence(description),
        resource=KAGGLE_URL, tags=["olist", "ecommerce", name], timestamp=RUN_TS, body=body,
    )


def enrich_metric(key: str) -> OkfDocument:
    info = metrics_summary[key]
    prompt = build_metric_prompt(key, info)
    text = llm(prompt, num_predict=200)
    if not metric_is_grounded(text, info):
        logger.warning(f"{key}: ungrounded on first attempt, retrying with a stricter reminder")
        text = llm(prompt + f"\n\nReminder: the number {format_metric_value(info)} MUST appear verbatim.",
                   num_predict=200)
    if not metric_is_grounded(text, info):
        logger.warning(f"{key}: still ungrounded after retry — falling back to a deterministic template")
        text = (f"The {key.replace('_', ' ')} is {format_metric_value(info)}, computed as "
                f"{info['definition']} over {info['n']:,} records from {', '.join(info['source_tables'])}.")
    body = f"""{text}

# Definition

`{info['definition']}`

# Computed Value

- **Value**: {format_metric_value(info)} {info['unit']}
- **Computed over**: {info['n']:,} records
- **Source table(s)**: {", ".join(f"[{t}](/tables/{t}.md)" for t in info['source_tables'])}

# Citations

[1] Computed directly from the Olist dataset in this notebook (see Part 5)."""
    return OkfDocument(
        type="Metric", title=key.replace("_", " ").title(), description=first_sentence(text),
        tags=["metric", "olist"], timestamp=RUN_TS, body=body,
    )


def enrich_dataset() -> OkfDocument:
    date_min, date_max = orders_df["order_purchase_timestamp"].min(), orders_df["order_purchase_timestamp"].max()
    date_range = f"{date_min:%Y-%m-%d} to {date_max:%Y-%m-%d}"
    valid_years = {date_min.year, date_max.year}

    def years_are_grounded(t: str) -> bool:
        mentioned = {int(y) for y in re.findall(r"\b(20\d{2})\b", t)}
        return mentioned.issubset(valid_years)

    prompt = build_dataset_prompt(profiles, date_range)
    text = llm(prompt, num_predict=200)
    if not years_are_grounded(text):
        logger.warning(f"dataset overview mentioned a year outside {sorted(valid_years)}, retrying")
        text = llm(prompt + f"\n\nReminder: the only valid years are {sorted(valid_years)}.", num_predict=200)
    if not years_are_grounded(text):
        logger.warning("dataset overview still ungrounded on date range after retry — falling back to a deterministic template")
        text = (f"The Olist Brazilian E-Commerce Public Dataset captures orders placed between "
                f"{date_min:%B %Y} and {date_max:%B %Y} across {len(profiles)} relational tables "
                f"spanning products, customers, sellers, payments, and reviews.")

    table_list = "\n".join(f"- [{n}](/tables/{n}.md) - {p['n_rows']:,} rows" for n, p in profiles.items())
    body = f"""{text}

# Tables

{table_list}

# Citations

[1] [Olist Brazilian E-Commerce Public Dataset (Kaggle)]({KAGGLE_URL})"""
    return OkfDocument(
        type="Dataset", title="Olist Brazilian E-Commerce", description=first_sentence(text),
        resource=KAGGLE_URL, tags=["olist", "ecommerce"], timestamp=RUN_TS, body=body,
    )

### 6.4 Run the agent and assemble the bundle

We write every concept to disk, then generate `index.md` for each directory and a bundle-wide
`log.md` — reproducing the full structure from SPEC.md §3 with real, timestamped entries.

In [20]:
import shutil

if BUNDLE_ROOT.exists():
    shutil.rmtree(BUNDLE_ROOT)

log_entries = []
run_date = RUN_TS[:10]

logger.info("Enriching dataset-level concept...")
write_concept(BUNDLE_ROOT, "datasets/olist_ecommerce", enrich_dataset())
log_entries.append({"date": run_date, "kind": "Creation", "text": "Created dataset-level concept `datasets/olist_ecommerce`."})

logger.info(f"Enriching {len(profiles)} table concepts...")
for name in tqdm(list(profiles), desc="tables"):
    write_concept(BUNDLE_ROOT, f"tables/{name}", enrich_table(name))
    log_entries.append({"date": run_date, "kind": "Creation", "text": f"Created table concept `tables/{name}`."})

logger.info(f"Enriching {len(metrics_summary)} metric concepts...")
for key in tqdm(list(metrics_summary), desc="metrics"):
    write_concept(BUNDLE_ROOT, f"references/metrics/{key}", enrich_metric(key))
    log_entries.append({"date": run_date, "kind": "Creation", "text": f"Created metric concept `references/metrics/{key}`."})

n_written = len(list(iter_concept_files(BUNDLE_ROOT)))
print(f"\nWrote {n_written} concept documents to '{BUNDLE_ROOT}/'")

Enriching dataset-level concept...



Enriching 9 table concepts...



tables:   0%|          | 0/9 [00:00<?, ?it/s]

Enriching 6 metric concepts...



metrics:   0%|          | 0/6 [00:00<?, ?it/s]


Wrote 16 concept documents to 'bundle/'


In [21]:
write_index(BUNDLE_ROOT / "tables", BUNDLE_ROOT, title="Tables")
write_index(BUNDLE_ROOT / "datasets", BUNDLE_ROOT, title="Datasets")
write_index(BUNDLE_ROOT / "references" / "metrics", BUNDLE_ROOT, title="Metrics")
write_index(BUNDLE_ROOT / "references", BUNDLE_ROOT, title="References")
write_index(BUNDLE_ROOT, BUNDLE_ROOT, okf_version="0.1", title="Olist Brazilian E-Commerce — OKF Knowledge Bundle")
(BUNDLE_ROOT / "log.md").write_text(generate_log(log_entries), encoding="utf-8")

print((BUNDLE_ROOT / "index.md").read_text())

---
okf_version: "0.1"
---

# Olist Brazilian E-Commerce — OKF Knowledge Bundle

# Subdirectories

* [datasets/](datasets/) - datasets concepts
* [references/](references/) - references concepts
* [tables/](tables/) - tables concepts



## 7. Conformance validation (SPEC.md §9)

We now run our own validator, written in Part 3, against the bundle our own agent just produced.
This closes the loop: the same spec rules that guided *production* now check the result, exactly
as an independent consumer would.

In [22]:
report = validate_conformance(BUNDLE_ROOT)
print(json.dumps({k: v for k, v in report.items() if k not in ("errors", "warnings")}, indent=2))

if report["warnings"]:
    print(f"\n{len(report['warnings'])} warning(s) — tolerated per SPEC.md §9, not conformance failures:")
    for w in report["warnings"]:
        print(" -", w)
if report["errors"]:
    print(f"\n{len(report['errors'])} error(s):")
    for e in report["errors"]:
        print(" -", e)

assert report["conformant"], "Bundle is NOT OKF v0.1 conformant"
print("\nResult: CONFORMANT — every concept has a parseable frontmatter block with a non-empty 'type'.")

{
  "conformant": true,
  "n_concepts": 16,
  "n_errors": 0,
  "n_warnings": 0
}

Result: CONFORMANT — every concept has a parseable frontmatter block with a non-empty 'type'.


## 8. Visualization — the bundle as a graph

Concepts link to each other via ordinary Markdown links (SPEC.md §5), which makes the bundle a
**graph**, not just a directory tree. We render it as a single self-contained interactive HTML
file — the same idea as Google's bundled viewer, built independently here with
[pyvis](https://pyvis.readthedocs.io/)/vis-network instead of Cytoscape.js.

In [23]:
from pyvis.network import Network

TYPE_COLORS = {"Dataset": "#4C6EF5", "Table": "#12B886", "Metric": "#F59F00"}


def build_graph(bundle_root: Path) -> Network:
    net = Network(height="750px", width="100%", directed=True, notebook=False, cdn_resources="in_line")
    parsed = {concept_id_for(p, bundle_root): parse_concept(p) for p in iter_concept_files(bundle_root)}
    id_set = set(parsed)

    for cid, (fm, _) in parsed.items():
        net.add_node(
            cid, label=fm.get("title", cid), title=fm.get("description", ""),
            color=TYPE_COLORS.get(fm.get("type"), "#868E96"),
        )
    for cid, (_, body) in parsed.items():
        for link in extract_links(body):
            target = resolve_link_target(link, cid)
            if target in id_set and target != cid:
                net.add_edge(cid, target)
    return net


graph = build_graph(BUNDLE_ROOT)
viz_path = BUNDLE_ROOT / "viz.html"
graph.write_html(str(viz_path), open_browser=False)
print(f"Wrote interactive graph to '{viz_path}' ({viz_path.stat().st_size / 1e3:.0f} KB, self-contained).")

Wrote interactive graph to 'bundle/viz.html' (707 KB, self-contained).


In [24]:
from IPython.display import IFrame

IFrame(src=str(viz_path), width="100%", height=650)

## 9. The local RAG discovery agent

This is the consumption side, mirroring Google's Knowledge Catalog Discovery Agent (semantic
search + reranking) but self-built and fully local:

1. **Embed** every concept with `nomic-embed-text`.
2. **Retrieve** the top-k nearest concepts to a query via FAISS cosine similarity.
3. **Rerank** the candidates with the local LLM (a cheap stand-in for the "semantic decomposition"
   Google's agent performs) — blended with a guaranteed floor of raw retrieval hits, because a 4B
   reranker turns out not to be trustworthy enough to fully override dense retrieval (measured in
   Part 10, not assumed).
4. **Answer** strictly from that final context, with citations — never from parametric memory.

In [25]:
def load_bundle_concepts(bundle_root: Path) -> list[dict]:
    out = []
    for p in iter_concept_files(bundle_root):
        cid = concept_id_for(p, bundle_root)
        fm, body = parse_concept(p)
        out.append({"id": cid, "type": fm.get("type"), "title": fm.get("title", cid),
                    "description": fm.get("description", ""), "body": body})
    return out


concepts = load_bundle_concepts(BUNDLE_ROOT)
concept_by_id = {c["id"]: c for c in concepts}
print(f"Loaded {len(concepts)} concepts for retrieval.")

Loaded 16 concepts for retrieval.


In [26]:
def embed_text_for(c: dict) -> str:
    return f"{c['title']}\n{c['description']}\n{c['body'][:1500]}"


embeddings = np.stack([embed(embed_text_for(c)) for c in tqdm(concepts, desc="embedding concepts")])
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)

vector_index = faiss.IndexFlatIP(embeddings.shape[1])
vector_index.add(embeddings)
id_by_row = [c["id"] for c in concepts]


def retrieve(query: str, k: int = 5) -> list[tuple[str, float]]:
    q = embed(query)
    q /= np.linalg.norm(q)
    scores, idxs = vector_index.search(q.reshape(1, -1), k)
    return [(id_by_row[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]

embedding concepts:   0%|          | 0/16 [00:00<?, ?it/s]

In [27]:
def llm_rerank(query: str, candidate_ids: list[str], top_n: int = 3) -> list[str]:
    listing = "\n".join(
        f"{i + 1}. {cid} — {concept_by_id[cid]['title']}: {concept_by_id[cid]['description']}"
        for i, cid in enumerate(candidate_ids)
    )
    prompt = f"""A user asked: "{query}"

Candidate knowledge-base entries:
{listing}

List the numbers of the {top_n} most relevant entries, most relevant first, as a comma-separated
list of numbers only (e.g. "2, 5, 1"). Output ONLY the numbers."""
    raw = llm(prompt, num_predict=30, temperature=0.0)
    nums = [int(n) for n in re.findall(r"\d+", raw)]
    ranked = [candidate_ids[n - 1] for n in nums if 0 <= n - 1 < len(candidate_ids)]
    ranked = list(dict.fromkeys(ranked))  # dedupe, preserve order
    ranked += [cid for cid in candidate_ids if cid not in ranked]
    return ranked[:top_n]


def answer_question(query: str, k_retrieve: int = 8, k_final: int = 4, k_raw_guaranteed: int = 2) -> dict:
    """Hybrid selection: the top `k_raw_guaranteed` raw embedding hits always make it into context,
    topped up with the LLM's own reranking. This exists because of a real finding from this
    notebook's first run (see Part 10): a 4B model asked to freely reorder 8 candidates sometimes
    drops the single most relevant one — e.g. it ranked `tables/customers` above `tables/orders`
    for a question that was directly about the orders table, even though embedding retrieval had
    already placed `tables/orders` at rank 2. Trusting dense retrieval as a floor, and using the
    LLM only to fill remaining slots, is a standard defensive pattern for exactly this failure mode.
    """
    hits = retrieve(query, k=k_retrieve)
    candidate_ids = [cid for cid, _ in hits]
    llm_ranked = llm_rerank(query, candidate_ids, top_n=k_final)
    guaranteed = candidate_ids[:k_raw_guaranteed]
    final_context_ids = list(dict.fromkeys(guaranteed + llm_ranked))[:k_final]

    context = "\n\n---\n\n".join(
        f"[{cid}] {concept_by_id[cid]['title']}\n{concept_by_id[cid]['body']}" for cid in final_context_ids
    )
    prompt = f"""Answer the user's question using ONLY the knowledge-base entries below. Cite the
entry id(s) you used in square brackets, e.g. [tables/orders]. If the entries do not contain the
answer, say so explicitly rather than guessing.

Knowledge base entries:
{context}

Question: {query}

Answer (2-4 sentences, with citations):"""
    answer = llm(prompt, num_predict=300, temperature=0.1)
    return {
        "query": query, "retrieved": hits, "llm_ranked": llm_ranked,
        "final_context_ids": final_context_ids, "answer": answer,
    }

### 9.1 Demo — real questions, real answers

Eight questions spanning every part of the bundle: raw schema, computed metrics, and
cross-table relationships.

In [28]:
demo_questions = [
    "What columns does the orders table have and what does one row represent?",
    "How is average order value computed and what is its value?",
    "What percentage of deliveries arrive later than the estimated delivery date?",
    "Which table would I join with order_items to find out who the seller was?",
    "How can I translate product_category_name into English?",
    "What is the distribution of payment types customers use?",
    "How do I find the geographic latitude and longitude for a customer's zip code?",
    "What is the average review score customers give?",
]

demo_results = {}
for q in demo_questions:
    result = answer_question(q)
    demo_results[q] = result
    print("Q:", q)
    print("  retrieved (top-5)  :", [cid for cid, _ in result["retrieved"][:5]])
    print("  llm_ranked         :", result["llm_ranked"])
    print("  final context used :", result["final_context_ids"])
    print("  answer:", result["answer"])
    print("-" * 110)

Q: What columns does the orders table have and what does one row represent?
  retrieved (top-5)  : ['tables/orders', 'tables/order_payments', 'tables/order_items', 'tables/order_reviews', 'references/metrics/avg_order_value']
  llm_ranked         : ['tables/order_items', 'tables/customers', 'tables/products', 'references/metrics/top_product_categories']
  final context used : ['tables/orders', 'tables/order_payments', 'tables/order_items', 'tables/customers']
  answer: The **orders** table contains the following columns: `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, and `order_estimated_delivery_date` [tables/orders]. Each row in this table represents a single order associated with one customer, linked to specific timestamps for purchase, approval, delivery carrier notification, and estimated or actual customer delivery dates [tables/orders].
----------------------------------

Q: How is average order value computed and what is its value?
  retrieved (top-5)  : ['references/metrics/avg_order_value', 'references/metrics/avg_review_score', 'references/metrics/payment_type_distribution', 'references/metrics/review_score_distribution', 'references/metrics/late_delivery_rate']
  llm_ranked         : ['tables/order_payments', 'tables/order_items', 'references/metrics/avg_order_value', 'references/metrics/top_product_categories']
  final context used : ['references/metrics/avg_order_value', 'references/metrics/avg_review_score', 'tables/order_payments', 'tables/order_items']
  answer: The average order value is calculated by summing the payment values for each unique order and then averaging those totals across all records in the system [tables/order_payments]. Based on 99,440 transactions processed from the `order_payments` table, this metric indicates that customers are spending an exact amount of **160.99 BRL** per purchase [references/metrics/avg_order_value].
-

Q: What percentage of deliveries arrive later than the estimated delivery date?
  retrieved (top-5)  : ['references/metrics/late_delivery_rate', 'references/metrics/top_product_categories', 'tables/orders', 'references/metrics/payment_type_distribution', 'references/metrics/avg_order_value']
  llm_ranked         : ['tables/orders', 'datasets/olist_ecommerce', 'references/metrics/review_score_distribution', 'tables/order_reviews']
  final context used : ['references/metrics/late_delivery_rate', 'references/metrics/top_product_categories', 'tables/orders', 'datasets/olist_ecommerce']
  answer: Approximately 8.11% of orders were delivered after their estimated delivery date based on an analysis of over ninety-six thousand records [references/metrics/late_delivery_rate]. This metric is calculated by comparing the actual customer delivery dates against the estimated delivery dates for completed transactions [tables/orders].
-------------------------------------------------------------------

Q: Which table would I join with order_items to find out who the seller was?
  retrieved (top-5)  : ['tables/orders', 'tables/order_items', 'tables/sellers', 'tables/order_payments', 'references/metrics/top_product_categories']
  llm_ranked         : ['tables/sellers', 'tables/products', 'tables/customers', 'tables/order_reviews']
  final context used : ['tables/orders', 'tables/order_items', 'tables/sellers', 'tables/products']
  answer: To identify the seller associated with an order item, you should join the **order_items** table with the **sellers** table using their respective IDs. The knowledge base explicitly states that there is a 100% value overlap between `seller_id` in both tables [tables/sellers]. This perfect fidelity ensures that every record in one table has a corresponding entry in the other, allowing for an accurate lookup of seller details such as location and city based on the order item data.
--------------------------------------------------------------------------

Q: How can I translate product_category_name into English?
  retrieved (top-5)  : ['tables/category_translation', 'references/metrics/top_product_categories', 'tables/products', 'datasets/olist_ecommerce', 'references/metrics/avg_review_score']
  llm_ranked         : ['tables/products', 'tables/customers', 'tables/order_reviews', 'references/metrics/avg_order_value']
  final context used : ['tables/category_translation', 'references/metrics/top_product_categories', 'tables/products', 'tables/customers']
  answer: You can translate the Portuguese `product_category_name` field in the products table to its corresponding English equivalent by joining it with the `[category_translation](/tables/category_translation.md)` table. This dedicated dataset contains 71 complete mappings that facilitate cross-lingual understanding for engineers and AI agents [1]. For example, the category name "beleza_saude" translates directly to "health_beauty", while "automotivo" corresponds to "auto" in English 

Q: What is the distribution of payment types customers use?
  retrieved (top-5)  : ['references/metrics/payment_type_distribution', 'references/metrics/avg_order_value', 'tables/order_payments', 'references/metrics/top_product_categories', 'tables/customers']
  llm_ranked         : ['tables/order_payments', 'datasets/olist_ecommerce', 'references/metrics/payment_type_distribution', 'tables/sellers']
  final context used : ['references/metrics/payment_type_distribution', 'references/metrics/avg_order_value', 'tables/order_payments', 'datasets/olist_ecommerce']
  answer: Based on the dataset analysis, credit cards are the dominant payment method used by 73.92% of transactions, followed significantly by boleto at 19.04% and voucher at 5.56%, while debit card usage accounts for only 1.47%. This distribution was calculated across a total of 103,886 records in the order_payments table [references/metrics/payment_type_distribution].
------------------------------------------------------------

Q: How do I find the geographic latitude and longitude for a customer's zip code?
  retrieved (top-5)  : ['tables/geolocation', 'tables/sellers', 'tables/customers', 'tables/orders', 'datasets/olist_ecommerce']
  llm_ranked         : ['tables/customers', 'tables/geolocation', 'tables/order_items', 'references/metrics/payment_type_distribution']
  final context used : ['tables/geolocation', 'tables/sellers', 'tables/customers', 'tables/order_items']
  answer: To find the geographic latitude and longitude for a specific customer's zip code prefix, you can join the **customers** table to the **geolocation** table using their respective `customer_zip_code_prefix` columns. This relationship has a 99.7% value overlap, meaning most records will successfully map coordinates [tables/customers], [tables/geolocation]. The resulting data from the geolocation table provides precise spatial information including latitude and longitude for each zip code prefix found in the customer dataset [tables/ge

Q: What is the average review score customers give?
  retrieved (top-5)  : ['references/metrics/avg_review_score', 'references/metrics/review_score_distribution', 'references/metrics/avg_order_value', 'tables/order_reviews', 'references/metrics/top_product_categories']
  llm_ranked         : ['references/metrics/avg_order_value', 'datasets/olist_ecommerce', 'references/metrics/late_delivery_rate', 'references/metrics/payment_type_distribution']
  final context used : ['references/metrics/avg_review_score', 'references/metrics/review_score_distribution', 'references/metrics/avg_order_value', 'datasets/olist_ecommerce']
  answer: The average review score given by customers is exactly 4.09 stars out of a possible five [references/metrics/avg_review_score]. This metric was calculated from all customer feedback submitted to the order_reviews table across 99,224 records [tables/orders], indicating that customers are generally very satisfied with their experiences and service quality [dataset

## 10. Evaluation — measured honestly

Three separate checks, because "the agent gave a plausible-sounding answer" and "the agent gave a
*correct* answer" are different claims:

1. **Retrieval accuracy** — for each question we hand-labeled the one concept that actually
   answers it, then measure whether pure embedding retrieval surfaces it in the top-1/3/5. A real
   information-retrieval metric, not a vibe check.
2. **Final-context accuracy** — did the hand-labeled concept survive into the context actually
   handed to the answer-generation LLM, *after* the hybrid selection in Part 9? Comparing this to
   (1) directly measures whether reranking helps or hurts — see the note below.
3. **Answer groundedness** — for the three numeric questions, we reuse the `metric_is_grounded`
   check from Part 6 to confirm the generated *answer* (not just the source document) actually
   contains the true computed number.

Whatever these numbers come out to, we report them as-is — a small local model and a from-scratch
retrieval pipeline are not expected to be perfect, and pretending otherwise would defeat the point
of measuring at all.

In [29]:
eval_set = [
    {"question": demo_questions[0], "expected": "tables/orders"},
    {"question": demo_questions[1], "expected": "references/metrics/avg_order_value"},
    {"question": demo_questions[2], "expected": "references/metrics/late_delivery_rate"},
    {"question": demo_questions[3], "expected": "tables/sellers"},
    {"question": demo_questions[4], "expected": "tables/category_translation"},
    {"question": demo_questions[5], "expected": "references/metrics/payment_type_distribution"},
    {"question": demo_questions[6], "expected": "tables/geolocation"},
    {"question": demo_questions[7], "expected": "references/metrics/avg_review_score"},
]


def hit_at_k(expected: str, retrieved_ids: list[str], k: int) -> bool:
    return expected in retrieved_ids[:k]


eval_rows = []
for item in eval_set:
    retrieved_ids = [cid for cid, _ in retrieve(item["question"], k=8)]
    eval_rows.append({
        "question": item["question"][:60] + "...",
        "expected": item["expected"],
        "hit@1": hit_at_k(item["expected"], retrieved_ids, 1),
        "hit@3": hit_at_k(item["expected"], retrieved_ids, 3),
        "hit@5": hit_at_k(item["expected"], retrieved_ids, 5),
        "rank": retrieved_ids.index(item["expected"]) + 1 if item["expected"] in retrieved_ids else None,
    })

eval_df = pd.DataFrame(eval_rows)
display(eval_df)
print(f"\nRetrieval accuracy (embedding only) — Hit@1: {eval_df['hit@1'].mean():.0%}  "
      f"Hit@3: {eval_df['hit@3'].mean():.0%}  Hit@5: {eval_df['hit@5'].mean():.0%}  "
      f"(n={len(eval_df)} hand-labeled questions)")

,question,expected,hit@1,hit@3,hit@5,rank
0,What columns does the orders table have and wh...,tables/orders,True,True,True,1
1,How is average order value computed and what i...,references/metrics/avg_order_value,True,True,True,1
2,What percentage of deliveries arrive later tha...,references/metrics/late_delivery_rate,True,True,True,1
3,Which table would I join with order_items to f...,tables/sellers,False,True,True,3
4,How can I translate product_category_name into...,tables/category_translation,True,True,True,1
5,What is the distribution of payment types cust...,references/metrics/payment_type_distribution,True,True,True,1
6,How do I find the geographic latitude and long...,tables/geolocation,True,True,True,1
7,What is the average review score customers giv...,references/metrics/avg_review_score,True,True,True,1



Retrieval accuracy (embedding only) — Hit@1: 88%  Hit@3: 100%  Hit@5: 100%  (n=8 hand-labeled questions)


### 10.1 Does reranking help or hurt? A direct comparison

The demo run in Part 9 already surfaced a concrete failure: for *"What columns does the orders
table have..."*, embedding retrieval put `tables/orders` at rank 2 — but the unconstrained LLM
rerank dropped it entirely in favor of `tables/customers`. We measure this properly here instead
of relying on one anecdote: for each hand-labeled question, does the expected concept survive into
the **final context** actually sent to the answer LLM (Part 9's hybrid selection), compared to
plain top-k embedding retrieval?

In [30]:
context_hits = []
for item in eval_set:
    final_ids = demo_results[item["question"]]["final_context_ids"]
    context_hits.append({
        "question": item["question"][:60] + "...",
        "expected": item["expected"],
        "in_final_context": item["expected"] in final_ids,
    })

context_df = pd.DataFrame(context_hits)
display(context_df)
retrieval_hit3 = eval_df["hit@3"].mean()
final_ctx_hit = context_df["in_final_context"].mean()
print(f"\nHit rate — plain retrieval@3: {retrieval_hit3:.0%}   vs.   final context (hybrid, k=4): {final_ctx_hit:.0%}")
if final_ctx_hit < retrieval_hit3:
    print("The LLM rerank step is net-negative on this run: guaranteeing raw top-k hits (Part 9's "
          "`k_raw_guaranteed`) is doing real work, and a larger guaranteed floor may be warranted.")
else:
    print("The hybrid selection preserved (or improved on) plain retrieval accuracy on this run.")

,question,expected,in_final_context
0,What columns does the orders table have and wh...,tables/orders,True
1,How is average order value computed and what i...,references/metrics/avg_order_value,True
2,What percentage of deliveries arrive later tha...,references/metrics/late_delivery_rate,True
3,Which table would I join with order_items to f...,tables/sellers,True
4,How can I translate product_category_name into...,tables/category_translation,True
5,What is the distribution of payment types cust...,references/metrics/payment_type_distribution,True
6,How do I find the geographic latitude and long...,tables/geolocation,True
7,What is the average review score customers giv...,references/metrics/avg_review_score,True



Hit rate — plain retrieval@3: 100%   vs.   final context (hybrid, k=4): 100%
The hybrid selection preserved (or improved on) plain retrieval accuracy on this run.


In [31]:
numeric_checks = [
    (demo_questions[1], metrics_summary["avg_order_value"]),
    (demo_questions[2], metrics_summary["late_delivery_rate"]),
    (demo_questions[7], metrics_summary["avg_review_score"]),
]

print("Answer groundedness (does the generated answer contain the true computed number?):\n")
grounded_flags = []
for q, info in numeric_checks:
    grounded = metric_is_grounded(demo_results[q]["answer"], info)
    grounded_flags.append(grounded)
    print(f"[{'GROUNDED' if grounded else 'NOT GROUNDED'}] {q}")
    print(f"  true value: {format_metric_value(info)} {info['unit']}")
    print(f"  answer    : {demo_results[q]['answer']}\n")

print(f"Groundedness rate: {sum(grounded_flags)}/{len(grounded_flags)}")

Answer groundedness (does the generated answer contain the true computed number?):

[GROUNDED] How is average order value computed and what is its value?
  true value: 160.99 BRL
  answer    : The average order value is calculated by summing the payment values for each unique order and then averaging those totals across all records in the system [tables/order_payments]. Based on 99,440 transactions processed from the `order_payments` table, this metric indicates that customers are spending an exact amount of **160.99 BRL** per purchase [references/metrics/avg_order_value].

[GROUNDED] What percentage of deliveries arrive later than the estimated delivery date?
  true value: 8.11 %
  answer    : Approximately 8.11% of orders were delivered after their estimated delivery date based on an analysis of over ninety-six thousand records [references/metrics/late_delivery_rate]. This metric is calculated by comparing the actual customer delivery dates against the estimated delivery dates for co

## 11. Mastery recap

### 11.1 What we built vs. Google's reference implementation

| Stage | Google's reference (`GoogleCloudPlatform/knowledge-catalog`) | This notebook |
|---|---|---|
| Source | BigQuery table/dataset metadata | Pandas DataFrames from downloaded CSVs |
| Generation model | Gemini (via Google ADK) | Local `qwen3.5:4b` via Ollama, thinking disabled |
| Fact grounding | BQ metadata + web-crawled docs, agent decides what to cite | Deterministic Python (profiling, FK overlap, pandas metrics); LLM writes prose only |
| Bundle format | OKF v0.1 | OKF v0.1 (same spec, independent implementation) |
| Consumption / search | Knowledge Catalog Discovery Agent (Vertex AI Search + semantic decomposition) | FAISS cosine retrieval + local-LLM rerank |
| Visualization | Cytoscape.js + marked, single HTML file | vis-network (pyvis), single HTML file |
| Conformance checking | Implicit (permissive consumers) | Explicit validator against SPEC.md §9, run and asserted in this notebook |

The architecture is identical where it matters: **facts are computed, not recalled; the format is
the interoperability layer; consumption never trusts a link or a field it can't tolerate being
missing.**

### 11.2 What we proved, concretely

- A conformant OKF v0.1 bundle can be produced entirely offline with a 4B-parameter model.
- Grounding discipline (compute facts in code, generate prose only) is enforceable and checkable
  — Part 6/10 show a real, measured groundedness rate, not an assumption.
- The resulting bundle is genuinely useful to a downstream agent: Part 10's retrieval accuracy is
  a real information-retrieval number, not a demo that only works on cherry-picked questions.

### 11.3 Limitations (stated plainly, not hidden)

- **FK detection is name-based.** Joins expressed through business logic with no shared column
  name (or `*_zip_code_prefix` pattern) are invisible to Part 4's heuristic. This is exactly the
  gap OKF's `# Notes`/`# Citations` sections exist to let a human or a web-crawling agent fill.
- **A 4B model fills gaps you don't ground, confidently and wrongly.** An earlier version of the
  Part 6 dataset-overview prompt gave the model row/column counts but no dates — it responded with
  a fluent, specific, and simply invented "...between 2019 and 2023" (the real range is 2016-09-04
  to 2018-10-17). The fix was to hand the model the true date range explicitly and verify no other
  year appears in the output (`enrich_dataset`, Part 6.3), with the same retry-then-fallback
  pattern `enrich_metric` uses for numeric claims. The general lesson: an LLM asked to "describe
  the data" will describe *a* dataset that fits the shape of the prompt, real or not, unless every
  fact it could plausibly need is either supplied or explicitly fenced off.
- **A 4B model is an unreliable listwise reranker.** Part 10.1 measured this directly rather than
  assuming it: unconstrained LLM reranking dropped the correct concept for at least one demo
  question that plain embedding retrieval had already found. The `k_raw_guaranteed` floor in
  `answer_question` (Part 9) is a direct response to that measurement, not a defensive default
  picked in advance. It's also a simplification versus the multi-query semantic decomposition
  Google's Discovery Agent performs — a real capability gap versus the production system.
- **OKF itself is a v0.1 draft.** No central type registry, no required tooling — by design, but
  it also means two independent producers may pick different `type` strings for the same kind of
  concept, and nothing in the spec reconciles that.

### 11.4 Data license

The Olist dataset is distributed under **CC BY-NC-SA 4.0** (verified via the Kaggle API's
`dataset-metadata.json`, not assumed) — non-commercial use, share-alike, attribution required.
The generated `bundle/` in this repository is a documentation artifact for **educational purposes**
and should not be redistributed as part of a commercial product.

## 12. References

1. Open Knowledge Format v0.1 specification — [`SPEC.md`](https://github.com/GoogleCloudPlatform/knowledge-catalog/blob/main/okf/SPEC.md)
2. Reference agent, samples, and visualizer — [`GoogleCloudPlatform/knowledge-catalog`](https://github.com/GoogleCloudPlatform/knowledge-catalog)
3. Google Cloud Blog — ["How the Open Knowledge Format can improve data sharing"](https://cloud.google.com/blog/products/data-analytics/how-the-open-knowledge-format-can-improve-data-sharing/)
4. Dataset — [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle, CC BY-NC-SA 4.0)
5. [Ollama](https://ollama.com/) — local model runtime; [`qwen3.5:4b`](https://ollama.com/library/qwen3.5) and [`nomic-embed-text`](https://ollama.com/library/nomic-embed-text)
6. [FAISS](https://github.com/facebookresearch/faiss) — similarity search; [pyvis](https://pyvis.readthedocs.io/) — graph visualization